# EXP-2026-003 / Q4-Q — transportability replication (quest49)

**상태: RESULT NOT RUN — full run도 PREP_DATA gate도 Q4-P 파생 분석도 아직 실행되지 않았다.**

spec: `experiments/specs/EXP-2026-003-q4q-transportability-replication.md` (사전 등록)

## 이 노트북이 표시하는 것
- 현재 mode (아래 config 셀; 기본값 `DESIGN` — GPU full run 아님)
- 사용할 데이터 경로/파일 ID와 data audit pass/fail
- full result가 없으면 `RESULT NOT RUN`

## 데이터 자산 (Drive 파일 ID)
- `mamba_data.npz` (MIT 1차): `1p3HvC_bnbiQlEanFOVIvVdejy60W0tho` — 99,871 beats · 44 records · DS1 22/DS2 22
- `ecg_multi.npz` (교차 검증): `1aSj_1jvS_W2iruVnORIG6DTVuHobzNzq`
- INCART annotation cache `.hea` (patient map): folder `1rNgzVlVYuiBDBfSjhHXm-Ksbw54nKfgM`
- Q4-P run bundle (ANALYZE 파생 분석 입력): folder `1qS8JxwlARByoZrJLMb6wxSIktQypiRTF`

## Colab 실행 순서
1. 셀 1(config)에서 MODE 선택 — 기본 `DESIGN`은 아무것도 실행하지 않는다
2. 셀 2: repo 준비 + 테스트 스위트(Q4-O 218 · Q4-P 87 · Q4-Q) 통과 확인
3. 셀 3: Drive mount + 경로 설정 (DESIGN이면 skip)
4. `PREP_DATA` → 셀 4 (data audit + ecg_multi 교차 검증 + INCART 75→32 map). **gate pass 전 FULL 금지**
5. `ANALYZE` → 셀 5 (Q4-P 무재학습 파생 분석 — 새 버전 경로에 저장, 원본 불변 검증)
6. `FULL` → 셀 6 (T4 GPU; DS2는 dev로 checkpoint 확정 후 딱 한 번 평가)
7. 셀 7: bundle 표·그림 표시 (없으면 RESULT NOT RUN 출력)

오류/gate 실패 시 조용한 fallback 없이 원인과 해결 명령을 출력하고 중단한다.

In [ ]:
# ── cell 1: mode config — 정확히 하나만 활성 ────────────────────────────
MODE = "DESIGN"   # "DESIGN" | "SMOKE" | "PREP_DATA" | "FULL" | "ANALYZE"

VALID_MODES = ("DESIGN", "SMOKE", "PREP_DATA", "FULL", "ANALYZE")
assert MODE in VALID_MODES, f"MODE must be exactly one of {VALID_MODES}"
assert sum(MODE == m for m in VALID_MODES) == 1
print(f"MODE = {MODE}")
if MODE == "DESIGN":
    print("DESIGN mode: nothing is executed. FULL RESULT NOT RUN.")

In [ ]:
# ── cell 2: 유효한 cwd 확보 → repo(절대 경로) → 강제 fresh import → tests ──
import os, subprocess, sys
if os.path.isdir("/content"):
    os.chdir("/content")          # 삭제된 cwd에 갇히지 않도록 항상 먼저 이동
REPO = ("/content/my-github-test" if os.path.isdir("/content")
        else os.path.abspath("my-github-test"))
URL = "https://github.com/ehdbddl06001-ui/my-github-test.git"
RUN_TESTS = True                  # 이 환경에서 한 번 통과했다면 False로 두면 빠름

ok = os.path.isdir(os.path.join(REPO, ".git"))
if ok:
    p = subprocess.run(["git", "-C", REPO, "pull", "--ff-only"],
                       capture_output=True, text=True)
    ok = (p.returncode == 0)
if not ok:
    subprocess.run(["rm", "-rf", REPO], check=False)
    r = subprocess.run(["git", "clone", URL, REPO],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr)
        raise SystemExit("git clone failed - see stderr above")
os.chdir(REPO)

# git pull은 디스크만 갱신한다 — sys.modules의 구버전을 pop하고 이 clone에서만
# fresh import 해야 stale-import가 원천 차단된다 (importlib.reload는 옛 중첩
# 경로의 파일을 다시 읽는 함정이 있어 쓰지 않는다).
for name in ("q4q_transportability_replication",
             "q4p_best_epoch_zero_diagnostic",
             "q4o_leakage_free_residual"):
    sys.modules.pop(name, None)
sys.path = [p for p in sys.path if "mit-bih" not in p]
sys.path.insert(0, os.path.join(REPO, "mit-bih"))
import q4o_leakage_free_residual as Q4O
import q4p_best_epoch_zero_diagnostic as QP
import q4q_transportability_replication as QQ

NEED_Q4Q = 6
assert QQ.MODULE_VERSION >= NEED_Q4Q, (
    f"stale q4q module {QQ.MODULE_VERSION} < {NEED_Q4Q} - "
    "Runtime > Restart runtime, then re-run from cell 1")
if RUN_TESTS:
    for suite in ("q4o_leakage_free_residual", "q4p_best_epoch_zero_diagnostic",
                  "q4q_transportability_replication"):
        rr = subprocess.run([sys.executable, f"mit-bih/test_{suite}.py"])
        assert rr.returncode == 0, f"test suite failed: {suite} - do not continue"
print("module ok:", QQ.self_check()["module_build"])

In [ ]:
# ── cell 3: Drive mount + 경로 (DESIGN이면 skip) ─────────────────────────
if MODE == "DESIGN":
    print("skip - DESIGN mode. RESULT NOT RUN.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    DATA_MAMBA = f"{DRIVE}/mitbih/mamba_data.npz"
    DATA_MULTI = f"{DRIVE}/mitbih/ecg_multi.npz"
    INCART_HEA = f"{DRIVE}/mitbih/raw_ann/incartdb"
    Q4P_RUN = (f"{DRIVE}/MedKOS/ecg-model/runs/"
               "20260808T1310_EXP-2026-002_q4p_best_epoch_zero_diagnostic")
    import time as _t
    TS = _t.strftime("%Y%m%dT%H%M")
    OUT_RUN = (f"{DRIVE}/MedKOS/ecg-model/runs/"
               f"{TS}_EXP-2026-003_q4q_transportability_replication")
    OUT_PREP = f"{DRIVE}/MedKOS/ecg-model/runs/{TS}_EXP-2026-003_prep_data"
    OUT_DERIVED = (f"{DRIVE}/MedKOS/ecg-model/runs/"
                   f"{TS}_EXP-2026-002_q4p_derived_analysis_v1")
    for p, need in ((DATA_MAMBA, MODE in ("PREP_DATA", "FULL")),
                    (Q4P_RUN, MODE == "ANALYZE")):
        if need and not os.path.exists(p):
            raise SystemExit(f"missing input: {p} - fix the path, do not guess")
    print("paths ready; outputs are NEW versioned dirs (no overwrite)")

In [ ]:
# ── cell 4: PREP_DATA — data audit gate (FULL 전 필수) ───────────────────
if MODE != "PREP_DATA":
    print(f"skip - MODE={MODE}. Data audit NOT RUN in this session.")
else:
    QQ.main(["--mode", "PREP_DATA", "--data", DATA_MAMBA,
             "--multi", DATA_MULTI, "--incart-hea", INCART_HEA,
             "--out", OUT_PREP])
    import json
    audit = json.load(open(f"{OUT_PREP}/data_audit.json"))
    cc = audit["cross_check"]
    print("cross_check pass:", cc.get("pass"))
    print("explanation:", cc.get("explanation", cc.get("fail_reasons")))
    print("S agreement:", cc.get("s_agreement"))
    for w in cc.get("warnings", []):
        print("warning:", w)
    print("incart_map:", audit.get("incart_map", "NOT RUN"))
    print("adapter gate:", audit["incart_adapter_audit"]["gate_pass"],
          "(INCART full run은 gate_pass=True 전 금지)")

In [ ]:
# ── cell 5: ANALYZE — Q4-P 무재학습 파생 분석 (사후 기전 분석) ───────────
if MODE != "ANALYZE":
    print(f"skip - MODE={MODE}. Q4-P derived analysis NOT RUN in this session.")
else:
    derived = QQ.q4p_derived_analysis(Q4P_RUN, OUT_DERIVED)
    rb = derived["did_cd_s2_minus_s0"]["record_bootstrap"]
    print(f"DiD (C-D) S2-S0: {rb['mean']:+.6f} "
          f"[{rb['ci_low']:+.6f}, {rb['ci_high']:+.6f}]")
    print("POST-HOC only - Q4-P의 사전 등록 판정(B3)은 바뀌지 않는다")
    from IPython.display import Image, display
    display(Image(f"{OUT_DERIVED}/derived_forest_waterfall.png"))

In [ ]:
# ── cell 6: FULL — MIT DS1->DS2 replication run (T4) ────────────────────
if MODE != "FULL":
    print(f"skip - MODE={MODE}. FULL RESULT NOT RUN.")
else:
    prep_ok = input("PREP_DATA gate를 통과했는가? (yes/no): ").strip()
    assert prep_ok == "yes", "PREP_DATA gate 통과 전 FULL 금지"
    QQ.main(["--mode", "FULL", "--data", DATA_MAMBA, "--out", OUT_RUN])
    print("bundle:", OUT_RUN)

In [ ]:
# ── cell 7: 결과 표시 (bundle 없으면 RESULT NOT RUN) ─────────────────────
import glob, json
run_dir = globals().get("OUT_RUN")
if not run_dir or not os.path.exists(f"{run_dir}/result.json"):
    print("RESULT NOT RUN - no measured bundle in this session.")
else:
    res = json.load(open(f"{run_dir}/result.json"))
    print("status:", res["status"], "| smoke:", res["smoke"])
    dm = res["decision_matrix"]
    print("mechanism pass:", dm["mechanism"]["pass"],
          "underpowered:", dm["mechanism"]["underpowered"])
    print("waveform confirmatory:", dm["waveform_specific"]["confirmatory"])
    print("utility pass:", dm["utility"]["pass"])
    print("action:", dm["action"])
    from IPython.display import Image, display
    for f in QQ.FIGURES:
        p = f"{run_dir}/figures/{f}"
        if os.path.exists(p):
            display(Image(p))

## 해석 경계 (사전 등록 요약)

- MIT-BIH는 과거 개발에 노출된 cohort — 이 실험은 **transportability replication**이지 untouched external confirmation이 아니다.
- DS2 결과를 보고 hyperparameter·preprocessing·selector를 바꾸지 않는다.
- 평균이 양수여도 CI가 0을 포함하면 **underpowered**로 기록한다. seed 추가는 patient 수 부족의 대체물이 아니다.
- INCART stage는 75→32 patient map과 adapter gate 통과 후에만, 동결 설계로 진행한다.